<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/Helixar-AI/gemma-cookbook/blob/main/FunctionGemma/Gemma_4_HDP_Agentic_Security/Gemma_4_HDP_Agentic_Security.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
</table>

# Securing Gemma 4 Agentic Workflows with HDP

**Author:** Asiri Dalugoda, Helixar Limited ([@asiridalugoda](https://github.com/asiridalugoda)) | [helixar.ai](https://helixar.ai)


## Before you begin

This notebook requires a GPU runtime. To enable GPU in Colab:
1. Go to **Runtime → Change runtime type**
2. Set **Hardware accelerator** to **GPU** (T4 is sufficient for E4B)
3. Click **Save**

You will also need a **Hugging Face token** to download Gemma 4 (gated model):
1. Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
2. Create a token with **Read** access
3. Accept the Gemma 4 model license at [huggingface.co/google/gemma-4-E4B-it](https://huggingface.co/google/gemma-4-E4B-it)
4. Run the cell below to authenticate

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

# Securing Gemma 4 Agentic Workflows with HDP

**Human Delegation Provenance (HDP)** is an open protocol that adds a cryptographic chain-of-custody to AI agent function calls — ensuring every tool invocation can be traced back to an authorized human principal.

This notebook demonstrates how to integrate HDP with Gemma 4's native function-calling capability to:

- **Verify** that Gemma 4's function calls were authorized by a human principal before execution
- **Classify** actions by irreversibility (read-only → irreversible → physical actuation)
- **Block** unauthorized or out-of-scope tool calls at the middleware layer
- **Audit** every decision with a pre-execution log

This is particularly relevant for Gemma 4 deployments on edge devices (Jetson Nano, Raspberry Pi) where the model may be directing physical actuators offline with no out-of-band authorization check.

**References:**
- HDP IETF draft: [draft-helixar-hdp-agentic-delegation-00](https://datatracker.ietf.org/doc/draft-helixar-hdp-agentic-delegation/)
- HDP-P (physical AI agents): [DOI 10.5281/ZENODO.19332440](https://doi.org/10.5281/ZENODO.19332440)
- Helixar: [helixar.ai](https://helixar.ai)

## Setup

In [ ]:
!pip install -q transformers torch cryptography

In [ ]:
# Download the middleware
!wget -q https://raw.githubusercontent.com/Helixar-AI/gemma4-hdp/main/hdp_middleware.py

from hdp_middleware import (
    HDPDelegationToken,
    HDPMiddleware,
    IrreversibilityClass,
    DEFAULT_TOOL_CLASS_MAP,
)
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
import json

## 1. Load Gemma 4

We use the 4B Effective model for this demo. For production agentic deployments, the 26B MoE or 31B Dense models are recommended.

In [ ]:
from transformers import pipeline

# For edge/robotics use cases: swap to google/gemma-4-E2B-it
MODEL_ID = "google/gemma-4-E4B-it"

pipe = pipeline(
    "text-generation",
    model=MODEL_ID,
    device_map="auto",
)

## 2. Define Tools

Gemma 4 uses structured JSON function-calling. We define a tool set spanning different IrreversibilityClasses to demonstrate the middleware's classification behaviour.

In [ ]:
TOOLS = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "City name"}
            },
            "required": ["location"]
        }
    },
    {
        "name": "send_email",
        "description": "Send an email to a recipient.",
        "parameters": {
            "type": "object",
            "properties": {
                "to": {"type": "string"},
                "subject": {"type": "string"},
                "body": {"type": "string"}
            },
            "required": ["to", "subject", "body"]
        }
    },
    {
        "name": "delete_file",
        "description": "Permanently delete a file by path.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string"}
            },
            "required": ["path"]
        }
    },
    {
        "name": "actuate_robot_arm",
        "description": "Command a robot arm to move to a target position.",
        "parameters": {
            "type": "object",
            "properties": {
                "joint_angles": {"type": "array", "items": {"type": "number"}},
                "force_limit_n": {"type": "number"}
            },
            "required": ["joint_angles"]
        }
    }
]

# Tools indexed by name for lookup
TOOL_REGISTRY = {t["name"]: t for t in TOOLS}
print(f"Registered {len(TOOLS)} tools")

## 3. Issue an HDP Delegation Token

The human principal generates an Ed25519 keypair and issues an HDT that specifies:
- Which tools the agent is permitted to call
- The maximum IrreversibilityClass the agent can act on
- The token's lifetime

In [ ]:
# Human principal generates their signing keypair
# In production: loaded from secure key storage (HSM, OS keychain, etc.)
principal_private_key = Ed25519PrivateKey.generate()
principal_public_key = principal_private_key.public_key()

# Issue an HDT authorizing the Gemma 4 agent to call weather queries
# and send emails (Class 0 and Class 2), but NOT delete files or actuate hardware
token = HDPDelegationToken.issue(
    principal_id="alice@example.com",
    agent_id="gemma4-agent-01",
    scope=["get_weather", "send_email"],
    max_class=IrreversibilityClass.CLASS_2,
    ttl_seconds=3600,
    private_key=principal_private_key,
)

print(json.dumps(token.to_dict(), indent=2))

## 4. Initialise the HDP Middleware

The middleware takes the principal's **public key** only — it verifies but cannot issue tokens.

In [ ]:
audit_log = []

# Confirmation callback for Class 2 (irreversible) actions.
# In production: this would invoke a push notification, SMS OTP,
# or hardware confirmation device to the human principal.
def require_human_confirmation(tool_name: str, parameters: dict) -> bool:
    print(f"\n⚠️  Class 2 action requested: {tool_name}")
    print(f"   Parameters: {json.dumps(parameters, indent=4)}")
    response = input("   Confirm? [y/N]: ").strip().lower()
    return response == "y"

middleware = HDPMiddleware(
    public_key=principal_public_key,
    tool_class_map=DEFAULT_TOOL_CLASS_MAP,
    confirmation_callback=require_human_confirmation,
    audit_log=audit_log,
)

print("HDP middleware initialised.")

## 5. Gemma 4 Function Call → HDP Gate → Tool Execution

This is the core integration pattern. Every function call Gemma 4 generates is passed through `middleware.gate()` before being forwarded to tool execution.

In [ ]:
# Simulated Gemma 4 function call outputs
# In production these come from parsing Gemma 4's structured JSON output
gemma_function_calls = [
    # ✅ Should ALLOW — Class 0, in scope
    {"name": "get_weather", "parameters": {"location": "Auckland"}},

    # ⚠️  Should CONFIRM then ALLOW — Class 2, in scope
    {"name": "send_email", "parameters": {
        "to": "bob@example.com",
        "subject": "Weekly report",
        "body": "Please find attached."
    }},

    # ❌ Should BLOCK — Class 2, NOT in HDT scope
    {"name": "delete_file", "parameters": {"path": "/data/important.csv"}},

    # ❌ Should BLOCK — Class 3, physical actuation
    {"name": "actuate_robot_arm", "parameters": {
        "joint_angles": [0.0, -1.57, 0.0, -1.57, 0.0, 0.0],
        "force_limit_n": 50.0
    }},
]

print("=" * 60)
print("HDP VERIFICATION RESULTS")
print("=" * 60)

for call in gemma_function_calls:
    result = middleware.gate(call, token)

## 6. Audit Log

Every decision is logged pre-execution. This is the HDP audit trail — a cryptographically linked record of what was authorized, by whom, and when.

In [ ]:
print("\nAUDIT LOG")
print("-" * 60)
for i, entry in enumerate(audit_log):
    status = "✅ ALLOWED" if entry.allowed else "❌ BLOCKED"
    print(f"{i+1}. {status} | {entry.tool_name} | {entry.action_class.name} | {entry.reason}")

## 7. Token Expiry and Scope Violation Demo

Demonstrate that expired tokens and out-of-scope calls are blocked regardless of the action class.

In [ ]:
import time

# Issue a token that's already expired
expired_token = HDPDelegationToken.issue(
    principal_id="alice@example.com",
    agent_id="gemma4-agent-01",
    scope=["get_weather"],
    max_class=IrreversibilityClass.CLASS_0,
    ttl_seconds=-1,   # expired immediately
    private_key=principal_private_key,
)

print("Testing expired token:")
middleware.gate({"name": "get_weather", "parameters": {"location": "Auckland"}}, expired_token)

print("\nTesting call outside HDT scope:")
middleware.gate({"name": "delete_file", "parameters": {"path": "/etc/passwd"}}, token)

## 8. Edge / Robotics Deployment (HDP-P)

For Gemma 4 E2B/E4B running on Jetson Nano or Raspberry Pi and directing physical actuators, use the HDP-P extension. The key additions are:

- **Embodiment context** — bind the token to a specific hardware ID
- **Policy attestation** — hash the deployed model weights into the token
- **Fleet delegation constraints** — prevent lateral movement across robot fleet
- **Pre-execution logging** — write audit records *before* actuator commands are issued

See the [HDP-P specification](https://doi.org/10.5281/ZENODO.19332440) for the full EDT extension structure.

In [ ]:
# Minimal HDP-P Embodied Delegation Token (EDT) extension example
# This shows how to attach physical constraints to an HDT

hdp_p_extension = {
    "hdp-p": {
        "version": "0.1",
        "embodiment": {
            "type": "mobile",
            "platform": "raspberry-pi-5",
            "hardware_id": "rpi-serial-XXXX",   # TPM-attested in production
            "workspace": "lab-zone-a"
        },
        "action_scope": {
            "permitted_actions": ["move_base", "read_sensor"],
            "excluded_zones": ["human-workspace"],
            "force_limit_n": 10.0,
            "max_velocity_ms": 0.5
        },
        "irreversibility": {
            "max_class": 1,                       # Class 1 max for this token
            "class2_requires_confirmation": True,
            "class3_prohibited": True
        },
        "policy_attestation": {
            "policy_hash": "sha256:abc123...",    # SHA-256 of deployed model weights
            "training_run_id": "gemma4-e2b-it",
            "sim_validated": True
        },
        "delegation_scope": {
            "fleet_delegation_permitted": False,   # No lateral movement
            "max_delegation_depth": 0
        }
    }
}

print("HDP-P EDT extension structure:")
print(json.dumps(hdp_p_extension, indent=2))

## Summary

| Layer | What it solves | Tool |
|---|---|---|
| Gemma 4 function calling | Model generates structured tool calls | `pipeline("text-generation")` |
| HDP middleware | Was this call authorized by a human? | `HDPMiddleware.gate()` |
| HDP-P EDT extension | Is this physical action within delegated bounds? | `hdp_p_extension` |
| Audit log | Pre-execution record of every decision | `audit_log` |

The full HDP specification (IETF draft), HDP-P companion paper, TypeScript SDK, and Python bindings are available at:

- **IETF draft:** https://datatracker.ietf.org/doc/draft-helixar-hdp-agentic-delegation/
- **HDP-P paper:** https://doi.org/10.5281/ZENODO.19332440
- **GitHub:** https://github.com/Helixar-AI
- **Site:** https://helixar.ai